In [2]:
from kfp import compiler, dsl, local
from kfp.dsl import (
    Input,
    Output,
    OutputPath,
    Dataset,
    Metrics,
    Model,
    component,
)
from typing import NamedTuple
import os
from dotenv import load_dotenv
from google.cloud import aiplatform
import json
load_dotenv(override=True)

True

In [3]:
PROJECT_ID = os.environ["PROJECT_ID"]

### Prepare Dataset

In [4]:
@component(
    base_image="python:3.12.9",
    packages_to_install=[
        "datasets==5.0.0",
    ],
)
def prepare_dataset(
    dataset_name: str,
    output_dataset: Output[Dataset],
):
    from datasets import load_dataset

    dataset = load_dataset(dataset_name)

    dataset.save_to_disk(output_dataset.path)

### Training

In [5]:
@component(
    base_image="python:3.12.9",
    packages_to_install=[
        "--extra-index-url=https://download.pytorch.org/whl/cu130",
        "torch==2.13.0+cu130",
        "torchvision==0.28.0+cu130",
        "transformers==5.14.1",
        "timm==1.0.26",
        "mlflow==3.11.1",
        "numpy==2.5.3",
        "Pillow==11.3.0",
        "datasets==5.0.0",
        "google-cloud-storage==3.13.0",
        "google-cloud-secret-manager==2.30.0",
        "psycopg2==2.9.9",
    ],
)
def train(
    project_id: str,
    dataset: Input[Dataset],
    parameters: str,
    gcs_ml_flow: str,
    gcs_package: str,
    dataset_name: str,
    wheel_name: str,
):
    import torch
    import sys
    import mlflow
    import json
    from datasets import load_from_disk

    # Dataset
    dataset = load_from_disk(dataset.path)
    train_dataset = dataset["train"]
    test_dataset = dataset["test"]

    print("Dataset:", dataset)
    print("Train length:", len(dataset["train"]))
    print("Test length:", len(dataset["test"]))    
    
    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device) 

    # Functions
    def check_environment():
        print("Python:", sys.version)
        print("PyTorch:", torch.__version__)
        print("CUDA:", torch.version.cuda)
        print("GPU available:", torch.cuda.is_available())
        print("MLflow version:", mlflow.__version__)

    def install_wheel():
        from google.cloud import storage
        import subprocess
        from urllib.parse import urlparse

        local_wheel = f"/tmp/{wheel_name}"
        gcs_wheel = f"{gcs_package.rstrip('/')}/{wheel_name}"
    
        print(f"Downloading wheel: {gcs_wheel}")
    
        parsed = urlparse(gcs_wheel)
        bucket_name = parsed.netloc
        blob_name = parsed.path.lstrip("/")
    
        client = storage.Client(project=project_id)
        bucket = client.bucket(bucket_name)
        blob = bucket.blob(blob_name)
    
        blob.download_to_filename(local_wheel)
    
        print("Downloaded wheel:", local_wheel)
    
        subprocess.run(
            [sys.executable, "-m", "pip", "install", local_wheel],
            check=True,
        )
    
        print("Wheel installed successfully.")   

    def get_servret_database_uri():
        from google.cloud import secretmanager
        client = secretmanager.SecretManagerServiceClient()
        name = f"projects/{project_id}/secrets/DATABASE_URI_ML_FLOW/versions/latest"
        response = client.access_secret_version(request={"name": name}) 
        return response.payload.data.decode("utf-8")

    # Parse Parameters
    parameters = json.loads(parameters)

    # Check Environment
    check_environment()

    # Install Wheel
    install_wheel() 
    
    # Get ML Flow Database Uri
    ml_flow_database_uri = get_servret_database_uri()

    # Train Function
    def run_train(args):
        # Main
        import torch
        import gc
        import copy
        
        model = None
        optimizer = None
        train_loader = None
        test_loader = None
        
        # 
        import torch
        import os
        import mlflow
        from pathlib import Path
        import copy   
        from transformers import AutoTokenizer
        from rf_donut_model.ml_flow_utils import set_experiment, get_run_name
        from rf_donut_model.seed import set_seed
        from rf_donut_model.data_loader import DonutDataset, get_data_loaders
        from rf_donut_model.save_load_model import get_model, save_model
        from rf_donut_model.train_eval import train_epoch, evaluate

        try:
            args = copy.deepcopy(args)
            checkpoint = args["checkpoint"]
            checkpoint_interval = args.get("checkpoint_interval", 20)
            seed = args.get("seed", 42)
            experiment_name = args.get("experiment_name", "Donut OCR-Free")
            run_name = args.get("run_name", "donut-ocr-free-run")
            repo_id = args.get("repo_id", None)
            
            # Dataset parameter
            num_workers = args.get("num_workers", 2)
            pin_memory = args.get("pin_memory", True)
            shuffle = args.get("shuffle", False)
            subset_size = args.get("subset_size", None)

            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
            print(f"Use Device:", device)

            # Set Seed
            set_seed(seed)
            
            # Set ML FLOW
            mlflow.set_tracking_uri(ml_flow_database_uri)
            set_experiment(experiment_name, gcs_ml_flow)
            
            # Set Tensors Folder
            Path("tensors").mkdir(parents=True, exist_ok=True)  

            ## Set Param           
            model, optimizer, configs, tags = get_model(checkpoint, device, repo_id)
            tags["device"] = str(device)
            tags["dataset_name"] = dataset_name
            configs["seed"] = seed
            configs["repo_id"] = repo_id

            # Main Param
            num_epochs = configs["train"]["num_epochs"]
            current_epoch = configs["train"]["epoch"]
            batch_size = configs["train"]["batch_size"]  
            
            # Dataset
            train_loader, test_loader = get_data_loaders(
                train_dataset = train_dataset,
                test_dataset = test_dataset,
                batch_size = batch_size,
                num_workers = num_workers,
                pin_memory = pin_memory,
                shuffle = shuffle,
                subset_size = subset_size,            
            )    
            
            # Utils
            tokenizer = AutoTokenizer.from_pretrained(
                "ridwanFatur98/invoice-doc-to-text",
                trust_remote_code=True,
            )            
            
            # Save checkpoint
            def save_checkpoint(state_name, val_predicted_results, train_predicted_results):
                if val_predicted_results is not None:
                    mlflow.log_dict({"predicted": val_predicted_results}, f"predictions/predicted-val-{state_name}.json")
                if train_predicted_results is not None:
                    mlflow.log_dict({"predicted": train_predicted_results}, f"predictions/predicted-train-{state_name}.json")
                model_path = f"tensors/checkpoint-{state_name}.pt"
                save_model(model, optimizer, configs, model_path)
                
                mlflow.log_artifact(model_path, artifact_path="model")
                artifact_path = f"{mlflow.get_artifact_uri()}/model/{model_path.split('/')[-1]}"
                print("Save Model:", artifact_path) 

            # Train
            with mlflow.start_run(run_name=get_run_name(run_name)):
                mlflow.log_params(configs)
                mlflow.set_tags(tags)
                mlflow.log_params({
                    "num_workers": num_workers,
                    "pin_memory": pin_memory,
                    "shuffle": shuffle,
                    "subset_size": subset_size if subset_size is not None else "full",
                    "checkpoint_interval": checkpoint_interval,
                })
                mlflow.log_params({
                    "train_dataset_size": len(train_loader.dataset),
                    "test_dataset_size": len(test_loader.dataset),
                    "train_num_batches": len(train_loader),
                    "test_num_batches": len(test_loader),
                })
                num_params = sum(p.numel() for p in model.parameters())
                mlflow.log_metric("num_params", num_params)
                
                ##### Save First Checkpoint ####
                save_checkpoint("init", None, None)
                
                for epoch in range(current_epoch, num_epochs):
                    ######### TRAIN ######
                    print("")
                    print("Epoch:", epoch)
                    print("Train") 
                    losses = train_epoch(
                        model, 
                        optimizer, 
                        train_loader, 
                        device,                       
                    )
                    avg_loss = sum(losses) / len(losses)
                    print("Train Loss | Loss:", avg_loss)
                    print("")                    

                    mlflow.log_metric("train_loss", avg_loss, step=epoch)  
                    
                    ######### Eval ######
                    print("Eval")   
                    
                    _, train_predicted_results = evaluate(model, train_loader, device, tokenizer)
                    losses, val_predicted_results = evaluate(model, test_loader, device, tokenizer)        
                    
                    avg_loss = sum(losses) / len(losses)
                    print("Val Loss | Loss:", avg_loss)
                    print("")
                        
                    mlflow.log_metric("val_loss", avg_loss, step=epoch)

                    ######### Update Configs ######
                    configs["train"]["epoch"] = epoch + 1
                
                    ######### Save Model ######
                    is_final = epoch == num_epochs - 1
                    is_checkpoint = epoch > 0 and epoch % checkpoint_interval == 0
                    
                    if is_checkpoint or is_final:
                        state_name = "final" if is_final else f"{epoch:04d}"
                        save_checkpoint(state_name, val_predicted_results, train_predicted_results)                       
        finally:
            print("Cleaning CUDA resources...")
            if model is not None:
                model.cpu()           
            del model
            del optimizer
            del train_loader
            del test_loader  
            gc.collect()   
            
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
                print(f"CUDA allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
                print(f"CUDA reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")           

    for index, param in enumerate(parameters, start=1):
        try:
            print(f"===== Train - {index} ======")
            run_train(param)
        except Exception as e:
            print(f"Error: {e}")

### Pipeline

In [6]:
@dsl.pipeline(name="pipeline")
def pipeline(
    project_id: str,
    parameters: str,
    gcs_ml_flow: str,
    gcs_package: str,
    dataset_name: str,
):
    prepare_dataset_task = prepare_dataset(
        dataset_name=dataset_name,
    )

    train_task = train(
        project_id=project_id,
        dataset=prepare_dataset_task.outputs["output_dataset"],
        parameters=parameters,
        gcs_ml_flow=gcs_ml_flow,
        gcs_package=gcs_package,
        dataset_name=dataset_name,
        wheel_name="rf_donut_model-0.1.0-py3-none-any.whl",
    )

    # train_task.set_cpu_limit("4")
    # train_task.set_memory_limit("16Gi")
    # train_task.set_accelerator_type("NVIDIA_TESLA_T4")
    # train_task.set_accelerator_limit(1)

### Compile

In [7]:
compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path="pipeline.yaml",
)

### Parameter

In [ ]:
checkpoints = {"test": "12"}
parameters = json.dumps(checkpoints)

### Deploy

In [26]:
GCS_PIPELINE = os.environ["GCS_PIPELINE"]
SERVICE_ACCOUNT = os.environ["SERVICE_ACCOUNT"]

PROJECT_ID = os.environ["PROJECT_ID"]

GCS_ML_FLOW = os.environ["GCS_ML_FLOW"]
GCS_PACKAGE = os.environ["GCS_PACKAGE"]

DATASET_NAME = "ridwanFatur98/synth-invoice"

In [28]:
job = aiplatform.PipelineJob(
    display_name="pipeline",
    template_path="pipeline.yaml",
    pipeline_root=GCS_PIPELINE,
    location="us-central1",
    parameter_values={
        "project_id": PROJECT_ID,
        "parameters": parameters,
        "gcs_ml_flow": GCS_ML_FLOW,
        "gcs_package": GCS_PACKAGE,
        "dataset_name": DATASET_NAME,        
    },
)

In [29]:
job.submit(service_account = SERVICE_ACCOUNT)

Creating PipelineJob
PipelineJob created. Resource name: projects/344969539300/locations/us-central1/pipelineJobs/pipeline-20261001085127
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/344969539300/locations/us-central1/pipelineJobs/pipeline-20261001085127')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/pipeline-20261001085127?project=344969539300


### Check Logs

In [22]:
from google.cloud import logging
client = logging.Client()

In [ ]:
# filter_ = '''

# '''
# logs = list(client.list_entries(filter_=filter_))
# len(logs)